# Diffusion-ES: does a more diverse initial pool help?

**Hypothesis:** some failures happen because the diffusion model's 64 initial samples all sit in one mode (one lane, one speed profile), so ES has nothing safe to refine. A more diverse, *cost-aware* first pool would give ES a safe starting point and make the planner react better.

**Protocol** (e450 world model + `margin_tiebreak`, the best config so far):

1. **Diagnose:** on scenarios that fail in every ES run, re-plan at the decisions before the failure and check whether the *initial* pool contained any safe plan.
2. **Intervene:** swap only the initial-pool sampler (`initial_pool=`), same seed, same reward, same ES loop.
3. **Closed loop:** success on the candidate set, baseline vs variants.

Initial-pool samplers in `es_debug`:
* `standard_pool`: production, 64 i.i.d. diffusion samples.
* `oversample_fps(factor)`: draw factor×64 and keep 64 by farthest-point sampling.
* `mcmc_pool(...)`: MCMC over smooth kinematic plans, with collision and offroad as the cost (Section 2).

In [ ]:
import os
# Torch WM + JAX planner share one GPU; must be set before JAX is imported.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import es_debug as esd

## 1. Configuration

In [ ]:
SUBSET_DIR = "/zfsauton/scratch/sbellad/Diffusion_ES_sim/es_failure_subset_n1500_seed0"
TFRECORD_DIR = f"{SUBSET_DIR}/tfrecords/training"
LANE_GRAPH_DIR = "/zfsauton/scratch/mineuih/waymax_rs/lane_graphs"

# ---- models: best config so far (e450 WM + margin_tiebreak) -----------------
DIFFUSION_CKPT = "/zfsauton/scratch/sbellad/train_diffusion/diffusion_baseline_baseline_20260810_144559/epoch_0500"
WM_CKPT = "/zfsauton/scratch/ishanj/world_model/ckpts_multifuture_gaussian/multiFuture_GaussianWorldModel_20261005_092950/epoch_0450.pt"
WM_BACKEND = "multifuture"
WM_PLANNING_MODE = "expected_independent"
REWARD = esd.margin_tiebreak_reward()

# ---- ES / sim (= slurm_es_fail1500_wm_mf.sh) --------------------------------
ES = dict(population_size=64, elite_size=8, num_iterations=4, resample_timesteps=5)
START_TIMESTEP = 10
REPLAN_INTERVAL = 10
SIM_AGENTS = "log"     # use "idm" for the reactivity cases
SEED = 0

### Candidates

Clean scenarios (not offroad/colliding at t=10, goal > 20 m away) that **fail in all four ES runs** (latest.pt binary / tiebreak, e450 tiebreak idm / log), i.e. not seed noise. Labels are the e450 tiebreak log outcome.

In [ ]:
CANDIDATES = {
    # collision in every ES run (47 total); 00009#388 / 00017#173 still reach the goal -> clean dodge tests
    "collision": [("00009", 388), ("00017", 173), ("00001", 243), ("00013", 133), ("00023", 131)],
    # offroad in every ES run although the logged ego never leaves the road (21 total)
    "offroad":   [("00011", 168), ("00091", 145), ("00092", 26), ("00106", 375), ("00124", 425)],
    # goal missed in every ES run (163 total)
    "goal_miss": [("00002", 0), ("00002", 70), ("00003", 331), ("00006", 57)],
    # collide only with IDM agents -> run with SIM_AGENTS="idm"
    "reactive":  [("00101", 237), ("00152", 363)],
}

def tfrecord_path(shard: str) -> str:
    return f"{TFRECORD_DIR}/training_tfexample.tfrecord-{shard}-of-01000"

## 2. MCMC proposal pool: formulation

Instead of perturbing diffusion samples, draw plans from a distribution that is (a) kinematically feasible and smooth by construction, (b) spread out in speed and curvature, and (c) tilted toward low collision / offroad cost under the same world model and lane graph that ES scores with.

### 2.1 Plan parametrization (smooth by construction)

Ego frame at the replan time: origin at the ego, $x$ forward, $\psi(0)=0$, current speed $v_0$, current curvature $\kappa_0$ (yaw change over the last 0.5 s / distance). Horizon $H$ (= 5 s, the diffusion horizon), $u = t/H \in [0,1]$.

Ego frame at the replan time also gives the current acceleration $a_0$ (speed change over the last 0.5 s).

Speed and curvature are Bernstein polynomials, $b_{j,n}(u) = \binom{n}{j}u^j(1-u)^{n-j}$. Speed has degree $n_v=4$ and curvature degree $n_\kappa=3$:

$$v(t) = \sum_{j=0}^{n_v} w_j\, b_{j,n_v}(u), \qquad \kappa(t) = \sum_{j=0}^{n_\kappa} c_j\, b_{j,n_\kappa}(u), \qquad w_0 = v_0,\;\; w_1 = v_0 + \tfrac{a_0 H}{n_v},\;\; c_0 = \kappa_0 .$$

Since $v'(0) = \tfrac{n_v}{H}(w_1 - w_0)$, pinning $w_1$ makes the plan start at the **current acceleration** as well as the current speed. Consecutive plans then join without an acceleration step at the replan, which removes one source of jerk.

The free parameters are $\theta = (w_2,w_3,w_4,\, c_1,c_2,c_3) \in \mathbb{R}^6$. A cubic $\kappa$ can change sign twice, so the space contains arcs, lane changes ($+\,-$), swerves that come back ($+\,-\,+$) and turns to the opposite side.

The trajectory is the unicycle rollout (trapezoid rule, 0.05 s), resampled at the model timestamps:

$$\dot\psi = \kappa\, v,\qquad \dot x = v\cos\psi,\qquad \dot y = v\sin\psi .$$

$\kappa$ and $v$ are polynomials, so $\psi$ is $C^\infty$: **no jitter is possible**, unlike perturbed diffusion samples.

### 2.2 Prior: kinematic feasibility

Gaussian around "keep doing what you are doing", with spread from realistic accelerations ($t_j = jH/n$ is the time of control point $j$):

$$w_j \sim \mathcal N\!\big(v_0,\; (\sigma_a t_j)^2\big), \qquad c_j \sim \mathcal N\!\Big(\kappa_0,\; \big(\sigma_{\text{lat}} / \max(v_0, v_{\text{floor}})^2\big)^2\Big).$$

So the speed at time $t$ is roughly $\mathcal N(v_0, (\sigma_a t)^2)$: the band widens with what is reachable. The curvature spread corresponds to a lateral acceleration of $\sigma_{\text{lat}}$, so it is wide at low speed and narrow at highway speed. Defaults are $\sigma_a = 1.5\ \text{m/s}^2$, $\sigma_{\text{lat}} = 2\ \text{m/s}^2$ and $v_{\text{floor}} = 3$ m/s.

Hard feasibility set $\mathcal F$. For the first three constraints the **convex-hull property** of Bernstein polynomials makes the bounds on control points exact for every $t$:

$$0 \le w_j \le v_{\max}, \qquad a_{\min} \le \tfrac{n}{H}(w_{j+1}-w_j) \le a_{\max}, \qquad |c_j| \le \kappa_{\max}, \qquad |\kappa(t)|\,v(t)^2 \le a_{\text{lat,max}}\ \text{(dense grid)}.$$

The derivative of a Bernstein polynomial is again Bernstein with coefficients $\tfrac{n}{H}(w_{j+1}-w_j)$. Defaults are $a_{\min}=-6$, $a_{\max}=3$ and $a_{\text{lat,max}}=5\ \text{m/s}^2$, $\kappa_{\max}=0.2\ \text{m}^{-1}$ (≈5 m radius), and $v_{\max}=25$ m/s.

Whitened coordinates: $\theta = \mu + \operatorname{diag}(\sigma)\, z$, so the prior is $p_0(z) \propto \mathcal N(z; 0, I)\,\mathbb 1[z \in \mathcal F]$.

### 2.3 Cost (energy)

The plan $\tau(z)$ is scored with the same WM futures and lane graph as ES:

$$E(\tau) = w_c\,(1 - C) + w_t\,(1 - T_{\text{hit}}) + w_o\,(1 - M) + w_d\,(1 - R) + w_g\,(1 - G) + w_j\, J$$

* $C$: collision score, the probability over the WM modes that the plan does not collide ($\texttt{expected\_independent}$; 1 = safe).
* $T_{\text{hit}}$: first contact time against the WM modes, mode-probability-weighted, earliest over agents, divided by the horizon (1 = no contact). It makes "collides later" cheaper than "collides now", so the chains keep moving even when everything collides.
* $M$: offroad margin score (0 offroad, 0.5–1 by clearance to the road edge).
* $R$: offroad depth/time tiebreak (1 on-road, smaller the earlier and deeper the plan leaves the road).
* $G$: **goal progress** relative to the pace needed to reach the episode goal (the logged position at step 90) in time. With $d_t$ the plan-to-goal distance:

  $$v_{\text{req}} = \frac{d_0}{\max(t_{\text{left}},\,H)}, \qquad G = \operatorname{clip}\!\Big(\frac{d_0 - \min_t d_t}{v_{\text{req}}\,H},\,0,\,1\Big), \qquad G = 1 \text{ if } \min_t d_t \le 2\text{ m}.$$

  A plan that keeps the required pace scores 1, and going faster earns nothing. This removes the incentive to crawl: without $G$, braking always helps the collision terms and costs nothing.
* $J$: **comfort**, $\operatorname{mean}_t\,(j_{\text{lon}}^2 + j_{\text{lat}}^2)/j_{\text{ref}}^2$ with $j_{\text{lon}} = \ddot v$, $j_{\text{lat}} = \tfrac{d}{dt}(\kappa v^2)$ and $j_{\text{ref}} = 2\ \text{m/s}^3$.

Defaults are $w_c = w_o = 4$, $w_t = w_d = w_g = 2$ and $w_j = 0.5$. Safety dominates: a plan that collides immediately costs about 6, while a plan that stops costs 2 in progress. A safe plan at the required pace has $E \approx 0$–$1$.

### 2.4 Target and sampler

$$\pi(z) \;\propto\; \mathcal N(z;0,I)\;\mathbb 1[z\in\mathcal F]\;\exp\!\big(-\lambda\,E(\tau(z))/T\big)$$

**Preconditioned Crank–Nicolson (pCN) Metropolis–Hastings** with $P_{\text{chain}} = 48$ parallel chains (one batched scoring call per step):

$$z' = \sqrt{1-\beta^2}\, z + \beta\,\xi,\quad \xi\sim\mathcal N(0,I), \qquad \alpha = \mathbb 1[z'\in\mathcal F]\cdot\min\!\big(1,\ e^{-\lambda (E(z') - E(z))/T}\big).$$

* **Why pCN:** the proposal is reversible with respect to the Gaussian prior, so the prior terms cancel in the acceptance ratio. Only the cost difference and the feasibility check remain, and proposals never drift out of the prior's scale.
* **Annealing:** $\lambda$ ramps 0 → 1 over the first half of the steps. Early steps explore the prior, later ones concentrate on low cost. The step size $\beta$ adapts during annealing toward 30% acceptance.
* **Initialization:** chains start from feasible prior draws in antithetic pairs $(z, -z)$, so every left swerve has a right-swerve partner and every faster plan a slower one.
* **Why this keeps diversity:** MCMC samples the distribution instead of optimizing it. Separate safe options (yield, swerve left, swerve right) each keep the chains that found them, and $T$ sets how strongly the cost is enforced: larger $T$ gives more spread, smaller $T$ gives greedier chains.

### 2.5 Into ES

The 48 chain states are converted to the diffusion's normalized space, $[x, y]/\text{ego\_range}$, $[v_x, v_y]/v_{\max}$, $\psi/\pi$. The other 16 pool slots ($\texttt{keep\_frac}=0.25$) are ordinary diffusion samples. Optionally the chain states are SDEdit-denoised ($\texttt{denoise=True}$, σ=1) to pull them toward the model's manifold. ES then runs unchanged, with re-noising at σ=1 to avoid the σ=2 jitter seen before.

### 2.6 Progress in the ES reward (optional)

ES scores with `margin_tiebreak`, which has no progress term either. `margin_tiebreak_reward(collision_eps, progress_weight=w)` uses:

$$\text{score} = C\cdot M\cdot\big((1-w) + w\,G\big) \;+\; 10^{-3}R \;+\; \epsilon_c\, T_{\text{hit}} .$$

Any collision-free on-road plan ($\ge 0.5(1-w)$) still beats any colliding one, and among safe plans the one that keeps pace wins. The variant `mcmc_ctb_prog` uses $w=0.2$.

Cost: 1 + `num_steps` scoring calls of 48 plans per decision (default 30). The WM forecast is cached per timestep, so each call is just collision/offroad geometry.

## 3. Load models

In [ ]:
policy, preprocess_cfg = esd.load_diffusion_policy(DIFFUSION_CKPT, seed=SEED)
wm = esd.load_world_model(WM_CKPT, backend=WM_BACKEND, planning_mode=WM_PLANNING_MODE)

baseline_planner = esd.DebugESPlanner(
    policy, preprocess_cfg,
    other_agent_forecast="wm", world_model_forecaster=wm,
    reward=REWARD, acquisition=esd.topk, initial_pool=esd.standard_pool, label="baseline",
    **ES,
)
MCMC = esd.mcmc_pool(num_steps=30, temperature=1.0, keep_frac=0.25)

# MCMC variants: ES re-noise at σ=1 (2.5) and the collision-time tiebreak in the ES reward, so ES
# still ranks plans when they all collide; the last one also rewards goal progress (2.6).
mcmc_kw = dict(initial_pool=MCMC, resample_noise_scale=1.0)
variants = {
    "baseline": baseline_planner,
    "fps_x4": baseline_planner.variant(label="fps_x4", initial_pool=esd.oversample_fps(4)),
    "mcmc_ctb": baseline_planner.variant(label="mcmc_ctb", **mcmc_kw,
                                         reward=esd.margin_tiebreak_reward(collision_eps=1e-2)),
    "mcmc_ctb_prog": baseline_planner.variant(label="mcmc_ctb_prog", **mcmc_kw,
                                              reward=esd.margin_tiebreak_reward(collision_eps=1e-2, progress_weight=0.2)),
}

## 4. One scenario

Runs the baseline closed loop, finds the first failure, and picks the replan steps leading up to it.

In [ ]:
CATEGORY, (SHARD, SCENARIO_IDX) = "collision", CANDIDATES["collision"][0]   # <- pick one

scenario = esd.load_scenario(tfrecord_path(SHARD), SCENARIO_IDX, lane_graph_dir=LANE_GRAPH_DIR)
base = esd.run_closed_loop(
    baseline_planner, scenario,
    start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
    sim_agents=SIM_AGENTS, seed=SEED, verbose=False,
)
print(base.report())
t_fail = esd.first_failure_t(base)
print("first failure t:", t_fail)

In [ ]:
esd.plot_timeline(base);

In [ ]:
# Decisions before the failure: the plan executing at t_fail and the three before it.
fail_anchor = base.anchor_for(t_fail if t_fail is not None else base.replan_steps[-1])
ANCHORS = [s for s in base.replan_steps if fail_anchor - 3 * REPLAN_INTERVAL <= s <= fail_anchor]
print("anchors:", ANCHORS)

### 4.0 When did the failure become unavoidable?

At the replan just before a collision, the obstacle can already be closer than the braking distance and too close for a lane change. Then no feasible plan avoids it, and every chain hits at the same time (identical energies, as at t=60 in 00009#388). The MCMC sampler only proposes feasible plans, so at each anchor it answers whether any feasible plan avoids the collision:

* `frac_collision_free > 0`: a feasible safe plan exists at this decision.
* `latest_contact_s` = horizon (5 s): at least one chain never hits.
* `latest_contact_s` ≈ 0.5 s for all chains: contact is already unavoidable, so look at an earlier anchor.

`A` is set to the **latest anchor that still has a collision-free chain**, the last point where the planner could have recovered.

In [ ]:
mcmc_probe = baseline_planner.variant(label="mcmc probe", initial_pool=esd.mcmc_pool(num_steps=30, keep_frac=0.0),
                                      num_iterations=0)
probe = {}
for anchor in ANCHORS:
    esd.replan_at(mcmc_probe, base, anchor, seed=SEED)
    probe[anchor] = esd.mcmc_summary(mcmc_probe.last_pool_info)
probe_df = pd.DataFrame(probe).T.rename_axis("anchor")
display(probe_df.round(2))
recoverable = [a for a in ANCHORS if probe[a]["frac_collision_free"] > 0]
A = recoverable[-1] if recoverable else ANCHORS[0]
print("inspecting A =", A, "" if recoverable else "(no anchor has a collision-free chain: earliest one)")

### 4.1 The initial pool: diffusion vs prior vs MCMC

Initial pool only (`num_iterations=0`, `keep_frac=0` so all 64 plans come from the sampler), at decision `A`. In the ES plots, orange marks agents the chosen plan hits with probability ≥ 5% (solid line = their WM top-1 future). Dots on that line and on the red plan mark the same instants, every 1 s, so you can see where they meet. The title gives the chosen plan's terms and per-agent P(hit).

* **diffusion**: production samples.
* **prior**: `num_steps=0`, plain draws from $p_0$. Shows the spread the kinematic prior alone gives.
* **mcmc**: 30 pCN steps.
* **mcmc + SDEdit**: the same chains, then denoised by the diffusion model at σ=1.

In the plot, the right-hand panels show each pool's speed and curvature profiles, colored by final energy (dark = low).

In [ ]:
pool_only = lambda label, pool: baseline_planner.variant(label=label, initial_pool=pool, num_iterations=0)
pool_traces, pool_info = {}, {}
for name, pool in {
    "diffusion": esd.standard_pool,
    "prior": esd.mcmc_pool(num_steps=0, keep_frac=0.0),
    "mcmc": esd.mcmc_pool(num_steps=30, keep_frac=0.0),
    "mcmc + SDEdit": esd.mcmc_pool(num_steps=30, keep_frac=0.0, denoise=True),
}.items():
    pl = pool_only(name, pool)
    pool_traces[name] = esd.replan_at(pl, base, A, seed=SEED)
    pool_info[name] = pl.last_pool_info

display(pd.DataFrame({k: esd.mcmc_summary(v) for k, v in pool_info.items() if v is not None}).T.round(2))
display(pd.DataFrame({k: esd.pool_stats(tr, 0) for k, tr in pool_traces.items()}).T[
    ["n_safe", "frac_collision_free", "frac_onroad", "best_score",
     "endpoint_std_across_m", "endpoint_std_along_m", "final_speed_std_mps", "heading_change_std_deg"]].round(2))

In [ ]:
for name, tr in pool_traces.items():
    print(name)
    esd.plot_es_iterations(tr, base.snapshots[A], ego_idx=scenario.ego_idx, view_m=35)
    if pool_info[name] is not None:
        esd.plot_mcmc(pool_info[name])
    plt.show()

### 4.2 Was there a safe plan, and does ES keep it?

All variants at all anchors, `iter 0` = initial pool, `iter 4` = after ES.

In [ ]:
rows, traces = [], {}
for anchor in ANCHORS:
    for name, planner in variants.items():
        tr = base.traces[anchor] if name == "baseline" else esd.replan_at(planner, base, anchor, seed=SEED)
        traces[(anchor, name)] = tr
        for it in (0, -1):
            rows.append({"anchor": anchor, "variant": name, **esd.pool_stats(tr, it)})
        rows[-1]["chosen_from"] = tr.source()
pool_df = pd.DataFrame(rows).set_index(["anchor", "variant", "iter"])
pool_df[["n_safe", "frac_safe", "frac_collision_free", "frac_onroad", "best_score",
         "endpoint_std_across_m", "endpoint_std_along_m", "final_speed_std_mps", "chosen_from"]].round(3)

In [ ]:
# One decision through the ES iterations (bold = elites, red = chosen plan).
V = "mcmc_ctb_prog"   # <- variant
esd.plot_es_iterations(traces[(A, V)], base.snapshots[A], ego_idx=scenario.ego_idx, view_m=35);

### 4.3 Closed loop

In [ ]:
# Closed loop from scratch with each variant (same seed).
runs = {"baseline": base}
for name, planner in variants.items():
    if name != "baseline":
        runs[name] = esd.run_closed_loop(
            planner, scenario,
            start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
            sim_agents=SIM_AGENTS, seed=SEED, verbose=False,
        )
        print(runs[name].report())
esd.plot_timeline(base, compare=[r for k, r in runs.items() if k != "baseline"]);

In [ ]:
V = "mcmc_ctb_prog"
esd.animate_closed_loop(runs[V], view_m=40, show_plans=True, show_elites=True)

### 4.4 Speed, acceleration and jerk

Executed ego speed (thick) and the speed profile planned at each replan (thin). Replan markers show where the chosen plan came from: `o` = descends from a diffusion sample, `^` = from an MCMC chain.

A speed jump at a replan where the new plan starts somewhere else means the planner switched plans. Before progress was rewarded, slow plans kept winning while danger was close, and once it passed the goal-conditioned diffusion samples (which drive at the logged pace) won, so the speed jumped. With progress in the cost and the acceleration pinned, the thin lines should join smoothly.

In [ ]:
esd.plot_speed_profile(list(runs.values()));

## 5. Sweep over the candidates

For each scenario:
1. Run the baseline closed loop.
2. Re-plan with every variant at the decisions before the failure, and record pool stats.
3. Run the closed loop with every variant.

Each scenario takes a few minutes per variant; trim `CANDIDATES` first if needed. Reactive cases use IDM agents.

In [ ]:
def run_case(category, shard, idx, seed=SEED):
    sim_agents = "idm" if category == "reactive" else SIM_AGENTS
    sc = esd.load_scenario(tfrecord_path(shard), idx, lane_graph_dir=LANE_GRAPH_DIR)
    loop_kw = dict(start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
                   sim_agents=sim_agents, seed=seed, verbose=False)
    b = esd.run_closed_loop(baseline_planner, sc, **loop_kw)
    t_fail = esd.first_failure_t(b)
    fa = b.anchor_for(t_fail if t_fail is not None else b.replan_steps[-1])
    anchors = [s for s in b.replan_steps if fa - 2 * REPLAN_INTERVAL <= s <= fa]
    pool_rows, loop_rows = [], []
    for name, planner in variants.items():
        for a in anchors:
            tr = b.traces[a] if name == "baseline" else esd.replan_at(planner, b, a, seed=seed)
            p0, pl = esd.pool_stats(tr, 0), esd.pool_stats(tr, -1)
            pool_rows.append({"category": category, "scenario": f"{shard}#{idx}", "variant": name, "anchor": a,
                              "n_safe_init": p0["n_safe"], "n_safe_final": pl["n_safe"],
                              "spread_init_m": p0["endpoint_pairwise_m"], "across_init_m": p0["endpoint_std_across_m"]})
        r = b if name == "baseline" else esd.run_closed_loop(planner, sc, **loop_kw)
        loop_rows.append({"category": category, "scenario": f"{shard}#{idx}", "variant": name,
                          "success": r.success, "overlap": r.overlap, "offroad": r.offroad,
                          "goal_reached": r.goal_reached, "min_goal_dist_m": round(r.min_goal_distance_m, 1),
                          "first_fail_t": esd.first_failure_t(r)})
    return pool_rows, loop_rows

POOL, LOOP = [], []
for category, cases in CANDIDATES.items():
    for shard, idx in cases:
        print(f"--- {category} {shard}#{idx}", flush=True)
        try:
            p, l = run_case(category, shard, idx)
        except Exception as e:  # keep the sweep going
            print("   failed:", repr(e))
            continue
        POOL += p; LOOP += l
        print("   " + "  ".join(f"{r['variant']}={'OK' if r['success'] else 'fail'}" for r in l), flush=True)

In [ ]:
# Diagnosis: did the *baseline* initial pool contain a safe plan at the decisions before the failure?
pool = pd.DataFrame(POOL)
diag = (pool[pool.variant == "baseline"]
        .groupby(["category", "scenario"])
        .agg(min_safe_init=("n_safe_init", "min"), min_safe_final=("n_safe_final", "min"),
             spread_init_m=("spread_init_m", "mean")))
diag["verdict"] = np.where(diag.min_safe_init == 0, "no safe plan sampled (diversity)",
                  np.where(diag.min_safe_final == 0, "safe plan lost by ES", "safe plan available (scoring/WM)"))
display(diag)
display(diag.verdict.value_counts())

In [ ]:
# Same decisions, per variant: safe plans in the initial pool and its spread.
display(pool.pivot_table(index=["category", "scenario", "anchor"], columns="variant",
                         values=["n_safe_init", "across_init_m"]).round(1))

In [ ]:
# Closed-loop outcome per variant.
loop = pd.DataFrame(LOOP)
display(loop.pivot_table(index=["category", "scenario"], columns="variant", values="success", aggfunc="first"))
display(loop.groupby("variant")[["success", "overlap", "offroad", "goal_reached"]].mean().round(2))

**Reading the result**

* The baseline fails every candidate by construction, so any success in a variant column is a recovery. Re-run with 2–3 seeds before trusting a single flip: the same config can flip on a different seed (00022#17 did).
* `mcmc_ctb` vs `mcmc_ctb_prog` shows what progress in the ES reward adds on top of the MCMC pool (which already has progress in its cost). `fps_x4` is the control that uses only the model's own modes.
* For goal misses, also check `min_goal_dist_m`: progress should shrink it even when the run still fails.
* If most verdicts are *safe plan available*, the next thing to fix is scoring or the WM forecast, not diversity.
* MCMC knobs: `temperature` (higher = more spread, lower = greedier), `num_steps`, `KinematicPrior(accel_sigma=..., lat_accel_sigma=...)` and `CostWeights(...)`.